In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/legacyagent")

assert ROOT.exists(), f"Project root not found: {ROOT}"

print("LEGACYAGENT — DAY 3 DATA AUDIT")
print("=" * 80)

for path in sorted(ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"FILE  {path.relative_to(ROOT)}  [{size_mb:.2f} MB]")
    elif path.is_dir():
        print(f"DIR   {path.relative_to(ROOT)}/")

Mounted at /content/drive
LEGACYAGENT — DAY 3 DATA AUDIT
FILE  01_dataset_baseline.ipynb  [0.12 MB]
FILE  02_baseline_evaluation.ipynb  [0.34 MB]
FILE  03_training_data_pipeline.ipynb  [0.04 MB]
FILE  case_list.json  [0.00 MB]
DIR   checkpoints/
FILE  dataset_report.json  [0.00 MB]
DIR   eval/
FILE  eval/entity_terms.json  [0.00 MB]
FILE  eval/evaluation_protocol.json  [0.03 MB]
FILE  eval/frozen_test_benchmark.jsonl  [0.74 MB]
DIR   eval_results/
DIR   manifests/
FILE  manifests/test_manifest.jsonl  [0.92 MB]
FILE  manifests/train_manifest.jsonl  [2.17 MB]
FILE  manifests/validation_manifest.jsonl  [0.54 MB]
DIR   processed/
FILE  qlora_compatibility.json  [0.00 MB]
DIR   raw_audio/
FILE  raw_audio/1977_76-709.mp3  [16.09 MB]
FILE  raw_audio/1977_76-709.wav  [128.13 MB]
FILE  raw_audio/1994_94-455.mp3  [12.21 MB]
FILE  raw_audio/1994_94-455.wav  [96.60 MB]
FILE  raw_audio/1995_94-1244.mp3  [13.25 MB]
FILE  raw_audio/1995_94-1244.wav  [104.92 MB]
FILE  raw_audio/1996_96-292.mp3  [12.25

In [2]:
import json
from pathlib import Path
from collections import Counter

ROOT = Path("/content/drive/MyDrive/legacyagent")
MANIFEST_DIR = ROOT / "manifests"

manifest_paths = {
    "train": MANIFEST_DIR / "train_manifest.jsonl",
    "validation": MANIFEST_DIR / "validation_manifest.jsonl",
    "test": MANIFEST_DIR / "test_manifest.jsonl",
}

loaded = {}

print("DAY 3 — EXISTING MANIFEST AUDIT")
print("=" * 80)

for split, path in manifest_paths.items():
    rows = []

    with open(path, "r") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    loaded[split] = rows

    print(f"\n{split.upper()}")
    print("-" * 40)
    print("Rows:", len(rows))
    print("Fields:", sorted(rows[0].keys()) if rows else [])

    # Show likely case identifier values
    for key in ["case_id", "case", "source_case"]:
        if rows and key in rows[0]:
            values = {str(row.get(key)) for row in rows}
            print(f"Unique {key}s:", len(values))

    print("\nFirst row:")
    print(json.dumps(rows[0], indent=2)[:3000])

print("\n" + "=" * 80)
print("READ-ONLY AUDIT COMPLETE")

DAY 3 — EXISTING MANIFEST AUDIT

TRAIN
----------------------------------------
Rows: 3702
Fields: ['audio_path', 'case_id', 'case_name', 'duration', 'end', 'needs_alignment', 'section_index', 'segment_id', 'speaker_id', 'speaker_name', 'speaker_role', 'split', 'start', 'text', 'turn_index']
Unique case_ids: 12

First row:
{
  "segment_id": "2011_10-704_000000",
  "case_id": "2011_10-704",
  "case_name": "Messerschmidt v. Millender",
  "audio_path": "/content/drive/MyDrive/legacyagent/raw_audio/2011_10-704.wav",
  "start": 0.0,
  "end": 9.022,
  "duration": 9.022,
  "text": "We will hear argument next in Case 10-704, Messerschmidt v. Millender. Mr. Coates.",
  "speaker_id": "john_g_roberts_jr",
  "speaker_name": "John G. Roberts, Jr.",
  "speaker_role": "scotus_justice",
  "section_index": 0,
  "turn_index": 0,
  "split": "train",
  "needs_alignment": false
}

VALIDATION
----------------------------------------
Rows: 903
Fields: ['audio_path', 'case_id', 'case_name', 'duration', 'end',

In [3]:
print("DAY 3 — SPLIT LEAKAGE AUDIT")
print("=" * 80)

train = loaded["train"]
validation = loaded["validation"]
test = loaded["test"]

def values(rows, key):
    return {str(row[key]) for row in rows}

train_cases = values(train, "case_id")
val_cases = values(validation, "case_id")
test_cases = values(test, "case_id")

train_segments = values(train, "segment_id")
val_segments = values(validation, "segment_id")
test_segments = values(test, "segment_id")

print("\nCASE COUNTS")
print("Train cases:", len(train_cases))
print("Validation cases:", len(val_cases))
print("Test cases:", len(test_cases))
print("Total unique cases:", len(train_cases | val_cases | test_cases))

print("\nCASE-LEVEL OVERLAP")
print("Train ∩ Validation:", train_cases & val_cases)
print("Train ∩ Test:", train_cases & test_cases)
print("Validation ∩ Test:", val_cases & test_cases)

print("\nSEGMENT-LEVEL OVERLAP")
print("Train ∩ Validation:", len(train_segments & val_segments))
print("Train ∩ Test:", len(train_segments & test_segments))
print("Validation ∩ Test:", len(val_segments & test_segments))

# Internal duplicate IDs
print("\nINTERNAL DUPLICATE SEGMENT IDs")
for name, rows in loaded.items():
    ids = [str(row["segment_id"]) for row in rows]
    duplicates = len(ids) - len(set(ids))
    print(f"{name}: {duplicates}")

case_leakage = (
    (train_cases & val_cases)
    or (train_cases & test_cases)
    or (val_cases & test_cases)
)

segment_leakage = (
    (train_segments & val_segments)
    or (train_segments & test_segments)
    or (val_segments & test_segments)
)

internal_duplicates = any(
    len([row["segment_id"] for row in rows])
    != len({row["segment_id"] for row in rows})
    for rows in loaded.values()
)

assert not case_leakage, "FAIL: Case leakage detected"
assert not segment_leakage, "FAIL: Segment leakage detected"
assert not internal_duplicates, "FAIL: Duplicate segment IDs detected"
assert len(train_cases | val_cases | test_cases) == 20, \
    "FAIL: Expected exactly 20 unique cases"

print("\n" + "=" * 80)
print("PASS — CASE-DISJOINT SPLIT VERIFIED")
print("PASS — ZERO SEGMENT LEAKAGE")
print("PASS — ZERO DUPLICATE SEGMENT IDs")
print("PASS — ALL 20 CASES ACCOUNTED FOR")
print("=" * 80)

DAY 3 — SPLIT LEAKAGE AUDIT

CASE COUNTS
Train cases: 12
Validation cases: 3
Test cases: 5
Total unique cases: 20

CASE-LEVEL OVERLAP
Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()

SEGMENT-LEVEL OVERLAP
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

INTERNAL DUPLICATE SEGMENT IDs
train: 0
validation: 0
test: 0

PASS — CASE-DISJOINT SPLIT VERIFIED
PASS — ZERO SEGMENT LEAKAGE
PASS — ZERO DUPLICATE SEGMENT IDs
PASS — ALL 20 CASES ACCOUNTED FOR


In [4]:
from pathlib import Path
from collections import Counter

print("DAY 3 — TRAINING ELIGIBILITY AUDIT")
print("=" * 80)

def audit_training_split(name, rows):
    empty_text = []
    missing_audio = []
    bad_timestamps = []
    needs_alignment = []
    under_3s = []
    over_30s = []

    durations = []
    total_seconds = 0.0

    for row in rows:
        segment_id = row["segment_id"]
        text = str(row.get("text", "")).strip()
        audio_path = Path(row["audio_path"])

        start = float(row["start"])
        end = float(row["end"])
        duration = float(row["duration"])

        durations.append(duration)
        total_seconds += duration

        if not text:
            empty_text.append(segment_id)

        if not audio_path.exists():
            missing_audio.append(str(audio_path))

        if start < 0 or end <= start or duration <= 0:
            bad_timestamps.append(segment_id)

        if bool(row.get("needs_alignment", False)):
            needs_alignment.append(segment_id)

        if duration < 3.0:
            under_3s.append(segment_id)

        if duration > 30.0:
            over_30s.append(segment_id)

    print(f"\n{name.upper()}")
    print("-" * 50)
    print("Total rows:", len(rows))
    print("Total manifest hours:", round(total_seconds / 3600, 2))
    print("Minimum duration:", round(min(durations), 3), "sec")
    print("Maximum duration:", round(max(durations), 3), "sec")
    print("Empty transcripts:", len(empty_text))
    print("Missing audio files:", len(set(missing_audio)))
    print("Invalid timestamps:", len(bad_timestamps))
    print("needs_alignment=True:", len(needs_alignment))
    print("Under 3 seconds:", len(under_3s))
    print("Over 30 seconds:", len(over_30s))

    return {
        "empty_text": empty_text,
        "missing_audio": missing_audio,
        "bad_timestamps": bad_timestamps,
        "needs_alignment": needs_alignment,
        "under_3s": under_3s,
        "over_30s": over_30s,
    }

train_audit = audit_training_split("train", loaded["train"])
val_audit = audit_training_split("validation", loaded["validation"])

print("\n" + "=" * 80)
print("AUDIT COMPLETE — NO DATA MODIFIED")
print("=" * 80)

DAY 3 — TRAINING ELIGIBILITY AUDIT

TRAIN
--------------------------------------------------
Total rows: 3702
Total manifest hours: 11.61
Minimum duration: 0.051 sec
Maximum duration: 80.505 sec
Empty transcripts: 0
Missing audio files: 0
Invalid timestamps: 0
needs_alignment=True: 64
Under 3 seconds: 929
Over 30 seconds: 64

VALIDATION
--------------------------------------------------
Total rows: 903
Total manifest hours: 3.01
Minimum duration: 0.25 sec
Maximum duration: 51.344 sec
Empty transcripts: 0
Missing audio files: 0
Invalid timestamps: 0
needs_alignment=True: 18
Under 3 seconds: 176
Over 30 seconds: 18

AUDIT COMPLETE — NO DATA MODIFIED


In [5]:
print("DAY 3 — DURATION + ALIGNMENT DIAGNOSTIC")
print("=" * 80)

def duration_diagnostic(name, rows):
    bins = {
        "<0.5s": 0,
        "0.5–1s": 0,
        "1–2s": 0,
        "2–3s": 0,
        "3–30s": 0,
        ">30s": 0,
    }

    alignment_ids = set()
    over_30_ids = set()

    for row in rows:
        d = float(row["duration"])
        sid = row["segment_id"]

        if d < 0.5:
            bins["<0.5s"] += 1
        elif d < 1.0:
            bins["0.5–1s"] += 1
        elif d < 2.0:
            bins["1–2s"] += 1
        elif d < 3.0:
            bins["2–3s"] += 1
        elif d <= 30.0:
            bins["3–30s"] += 1
        else:
            bins[">30s"] += 1

        if row.get("needs_alignment", False):
            alignment_ids.add(sid)

        if d > 30.0:
            over_30_ids.add(sid)

    print(f"\n{name.upper()}")
    print("-" * 50)

    for label, count in bins.items():
        print(f"{label:10} {count}")

    print("\nAlignment relationship:")
    print("needs_alignment:", len(alignment_ids))
    print("over 30s:", len(over_30_ids))
    print("both:", len(alignment_ids & over_30_ids))
    print("alignment-only:", len(alignment_ids - over_30_ids))
    print("over-30-only:", len(over_30_ids - alignment_ids))

    shortest = sorted(rows, key=lambda r: float(r["duration"]))[:10]

    print("\n10 shortest segments:")
    for row in shortest:
        print(
            f'{row["duration"]:7.3f}s | '
            f'{row["segment_id"]} | '
            f'{row["text"][:100]!r}'
        )

duration_diagnostic("train", loaded["train"])
duration_diagnostic("validation", loaded["validation"])

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE — NO DATA MODIFIED")
print("=" * 80)

DAY 3 — DURATION + ALIGNMENT DIAGNOSTIC

TRAIN
--------------------------------------------------
<0.5s      114
0.5–1s     258
1–2s       357
2–3s       200
3–30s      2709
>30s       64

Alignment relationship:
needs_alignment: 64
over 30s: 64
both: 64
alignment-only: 0
over-30-only: 0

10 shortest segments:
  0.051s | 2013_13-483_000189 | 'Neither one.'
  0.062s | 2013_12-1117_000134 | 'That--'
  0.120s | 2016_15-1358_000064 | 'Yeah, yeah.'
  0.167s | 2011_10-704_000248 | 'Right.'
  0.194s | 2001_01-309_000362 | 'The...'
  0.219s | 2001_01-309_000359 | 'thi-'
  0.220s | 2016_15-1358_000096 | "That's true."
  0.234s | 1995_94-1244_000323 | 'Why?'
  0.249s | 2011_10-1018_000037 | 'No--'
  0.250s | 2011_10-704_000082 | 'Of course--'

VALIDATION
--------------------------------------------------
<0.5s      11
0.5–1s     48
1–2s       58
2–3s       59
3–30s      709
>30s       18

Alignment relationship:
needs_alignment: 18
over 30s: 18
both: 18
alignment-only: 0
over-30-only: 0

10 shor

In [6]:
import re
import statistics

print("DAY 3 — TEXT / DURATION CONSISTENCY AUDIT")
print("=" * 80)

def word_count(text):
    return len(re.findall(r"\b[\w']+\b", str(text)))

def consistency_audit(name, rows):
    records = []

    for row in rows:
        duration = float(row["duration"])
        words = word_count(row["text"])
        words_per_second = words / duration if duration > 0 else float("inf")

        records.append({
            "segment_id": row["segment_id"],
            "duration": duration,
            "words": words,
            "wps": words_per_second,
            "text": row["text"],
            "needs_alignment": bool(row.get("needs_alignment", False)),
        })

    eligible_for_rate_check = [
        r for r in records
        if not r["needs_alignment"] and r["duration"] <= 30.0
    ]

    wps_values = [r["wps"] for r in eligible_for_rate_check]

    print(f"\n{name.upper()}")
    print("-" * 60)
    print("Rows checked:", len(eligible_for_rate_check))
    print("Median words/sec:", round(statistics.median(wps_values), 2))

    for threshold in [3, 4, 5, 6, 8, 10]:
        count = sum(r["wps"] > threshold for r in eligible_for_rate_check)
        print(f"> {threshold:2} words/sec: {count}")

    print("\n20 highest words/sec rows:")
    for r in sorted(
        eligible_for_rate_check,
        key=lambda x: x["wps"],
        reverse=True
    )[:20]:
        print(
            f'{r["wps"]:7.2f} w/s | '
            f'{r["duration"]:6.3f}s | '
            f'{r["words"]:2} words | '
            f'{r["segment_id"]} | '
            f'{r["text"][:120]!r}'
        )

consistency_audit("train", loaded["train"])
consistency_audit("validation", loaded["validation"])

print("\n" + "=" * 80)
print("AUDIT COMPLETE — NO DATA MODIFIED")
print("=" * 80)

DAY 3 — TEXT / DURATION CONSISTENCY AUDIT

TRAIN
------------------------------------------------------------
Rows checked: 3638
Median words/sec: 2.92
>  3 words/sec: 1655
>  4 words/sec: 435
>  5 words/sec: 174
>  6 words/sec: 75
>  8 words/sec: 25
> 10 words/sec: 11

20 highest words/sec rows:
  39.22 w/s |  0.051s |  2 words | 2013_13-483_000189 | 'Neither one.'
  16.67 w/s |  0.120s |  2 words | 2016_15-1358_000064 | 'Yeah, yeah.'
  16.13 w/s |  0.062s |  1 words | 2013_12-1117_000134 | 'That--'
  14.98 w/s |  0.267s |  4 words | 2011_10-704_000178 | '--No, I understand, but--'
  14.06 w/s |  0.640s |  9 words | 2006_06-278_000313 | 'Even though the law required him to be there?'
  11.40 w/s |  0.702s |  8 words | 1995_94-1244_000142 | "Well, that's why I asked. I don't, either."
  11.33 w/s |  0.353s |  4 words | 2013_12-1117_000121 | '--between 3 and 12.'
  11.06 w/s |  0.452s |  5 words | 2006_06-278_000340 | 'In the afternoon he was.'
  10.68 w/s |  0.749s |  8 words | 2013_13